# Introduction: Why Probability for Machine Learning

## What's covered

- Why probability is the third foundation of ML (after linear algebra and calculus) — the **four jobs** probability does in every ML system
- The roadmap through this repo — 8 notebooks from axioms to A/B testing
- **Sample spaces** and **events** — the language probability is built from
- The three **Kolmogorov axioms** that everything else follows from
- **Conditional probability** and **Bayes' theorem** — the engine of every probabilistic model
- **Independence** — when two events don't tell you anything about each other


## Why probability for machine learning

Linear algebra and calculus gave us the algorithms. **Probability is the language we use to talk about uncertainty**, and ML is fundamentally a discipline of reasoning under uncertainty. Every prediction comes with a confidence; every model has noise; every dataset is a sample from a larger world we never fully see.

Probability does four jobs in ML — and the eight notebooks of this repo each serve at least one of them:

**1. Model the data-generating process.** Every dataset is treated as samples from some unknown distribution `p(x, y)`. Whether you train a regression or a transformer, you are implicitly or explicitly fitting a probabilistic model. Naive Bayes, logistic regression, Gaussian mixture models, VAEs, diffusion models — all directly probabilistic. Even "non-probabilistic" models like SVMs have probabilistic interpretations.

**2. Quantify uncertainty in predictions.** A classifier doesn't just say "cat" — it says "92% cat, 5% dog, 3% other." Those probabilities matter for calibration, for active learning (which examples to label next?), for safety-critical applications, and for any decision-making downstream of the model.

**3. Justify the loss functions you train with.** Mean squared error = MLE under Gaussian noise. Cross-entropy = negative log-likelihood for categorical outcomes. The L1 loss = MLE under Laplace noise. Every common loss has a probabilistic story underneath. Once you see it, you understand *why* the loss is what it is, and when it's the wrong choice.

**4. Reason about generalization and statistical tests.** How confident are we that model A beats model B? How big a test set do we need? The Central Limit Theorem, confidence intervals, and hypothesis testing are how you answer these. A/B testing — which every product team runs — is applied statistics.

Memorize those four jobs. They are the entire reason probability shows up in ML.


## The roadmap

Eight notebooks build from first axioms up to the statistical tools you'll use on the job.

1. **This notebook** — probability axioms, conditional probability, Bayes' theorem.
2. **Random variables and distributions** — how we package uncertainty into mathematical objects; the staple distributions (Bernoulli, Categorical, Poisson, Gaussian, etc.).
3. **Expectation, variance, and moments** — the summary statistics of distributions; identities you'll use constantly.
4. **Joint, marginal, and conditional distributions** — multiple random variables at once; the chain rule of probability.
5. **The Gaussian distribution** — the most-used distribution in ML, in both its scalar and multivariate forms.
6. **Maximum likelihood and Bayesian inference** — how to *learn* from data, both frequentist and Bayesian styles.
7. **Information theory** — entropy, cross-entropy, KL divergence; the math of classification losses and variational inference.
8. **Sampling, the Central Limit Theorem, and hypothesis testing** — the LLN, the CLT, Monte Carlo, A/B testing.

By the end you'll be fluent in the language of probabilistic modeling, comfortable computing likelihoods and posteriors, and able to design and analyze A/B tests.


## Sample spaces and events

Before we talk about probabilities, we need objects to assign probabilities to.

A **sample space**, written `\Omega` (capital omega), is the set of all possible outcomes of an experiment. For a coin flip, `\Omega = \{H, T\}`. For a die roll, `\Omega = \{1, 2, 3, 4, 5, 6\}`. For "tomorrow's temperature in Mumbai," `\Omega = \mathbb{R}` (or some interval thereof). The choice of `\Omega` is part of *modeling* — you pick the sample space that matters for the question you're asking.

An **event** is a subset of `\Omega`. The event "the die comes up even" is `\{2, 4, 6\} \subset \Omega`. The event "the temperature is between 28 and 32 degrees" is the interval `[28, 32] \subset \mathbb{R}`. Events are the things we want probabilities of.

Two important special events: `\Omega` itself (the certain event — *something* must happen) and `\emptyset` (the impossible event — none of the outcomes occur).

**Operations on events.** Events combine with set operations, which mirror logical operations.

| Set operation | Logical meaning | Notation |
|---|---|---|
| `A \cup B` | "A or B (or both)" | union |
| `A \cap B` | "A and B" | intersection |
| `A^c` (or `\bar{A}`) | "not A" | complement |
| `A \subseteq B` | "A implies B" | subset |

For example: rolling a die, let `A = \{1, 2, 3\}` ("low") and `B = \{2, 4, 6\}` ("even"). Then `A \cap B = \{2\}` ("low and even") and `A \cup B = \{1, 2, 3, 4, 6\}` ("low or even").


## The Kolmogorov axioms

A **probability** is a function `P` that takes an event and returns a number in `[0, 1]`. The number is the "weight" of that event — how often it happens, or how much we believe in it.

Andrey Kolmogorov realized in 1933 that all of probability theory follows from just **three rules**:

**Axiom 1 — Non-negativity.** `P(A) \geq 0` for every event `A`. Probabilities can't be negative.

**Axiom 2 — Normalization.** `P(\Omega) = 1`. *Something* happens with probability 1.

**Axiom 3 — Countable additivity.** For any *disjoint* (non-overlapping) events `A_1, A_2, \dots`:

$$
P\left(\bigcup_i A_i\right) = \sum_i P(A_i)
$$

That's it. Everything we derive — Bayes' theorem, the law of large numbers, the central limit theorem — follows from these three axioms.

**Immediate consequences.**

- `P(\emptyset) = 0` — the impossible event has probability 0.
- `P(A^c) = 1 - P(A)` — probabilities of complements sum to 1.
- `P(A \cup B) = P(A) + P(B) - P(A \cap B)` — the **inclusion-exclusion** formula, for *overlapping* events. (For disjoint events it reduces to the simpler axiom 3.)
- `0 \leq P(A) \leq 1` for every `A`.

We rarely cite the axioms in ML practice. But every probabilistic argument you'll see rests on them.


In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# Simulate rolling a fair die 100,000 times and verify the axioms empirically
n = 100_000
rolls = rng.integers(1, 7, size=n)

# Define events
A = (rolls <= 3)              # "low"      = {1, 2, 3}
B = (rolls % 2 == 0)          # "even"     = {2, 4, 6}

P_A = np.mean(A)
P_B = np.mean(B)
P_A_and_B = np.mean(A & B)
P_A_or_B  = np.mean(A | B)
P_not_A   = np.mean(~A)

print(f"P(A = low)         ~ {P_A:.4f}   (analytic 3/6 = 0.5)")
print(f"P(B = even)        ~ {P_B:.4f}   (analytic 3/6 = 0.5)")
print(f"P(A and B = even≤3)~ {P_A_and_B:.4f}   (analytic 1/6 ≈ 0.1667)")
print(f"P(A or B)          ~ {P_A_or_B:.4f}   (analytic 5/6 ≈ 0.8333)")
print(f"P(not A)           ~ {P_not_A:.4f}   (analytic 3/6 = 0.5)")

# Inclusion-exclusion check: P(A ∪ B) = P(A) + P(B) − P(A ∩ B)
print(f"\ninclusion-exclusion: P(A) + P(B) - P(A∩B) = {P_A + P_B - P_A_and_B:.4f}")
print(f"                      P(A∪B) directly     = {P_A_or_B:.4f}")


## Conditional probability

The probability of `A` *given that we already know* `B` occurred is written `P(A | B)`, read **"probability of A given B."** The definition:

$$
P(A | B) = \frac{P(A \cap B)}{P(B)}, \qquad \text{provided } P(B) > 0
$$

Geometrically: we're restricting our universe to outcomes where `B` happened, and asking what fraction of those *also* have `A`. The denominator `P(B)` renormalizes — within this restricted universe, `B` happens with probability 1.

**Example.** Roll a die. `P(\text{1}) = 1/6`. But `P(\text{1} | \text{odd}) = (1/6) / (1/2) = 1/3`. Once we know the result is odd, the only possible outcomes are `\{1, 3, 5\}`, and 1 is one of three.

**The product rule** (rearranging the definition):

$$
P(A \cap B) = P(A | B) \, P(B) = P(B | A) \, P(A)
$$

This pair of equalities is the workhorse of all probabilistic reasoning. It's also the seed of Bayes' theorem.

**ML angle.** Almost every prediction you make is a conditional probability: `P(y = \text{cat} | x = \text{image})`. Classifiers output conditional distributions. Language models output `P(\text{next token} | \text{previous tokens})`. The whole apparatus of supervised learning is *estimating conditional distributions*.


In [ ]:
# Verify conditional probability: P(roll = 1 | odd) = 1/3
n = 200_000
rolls = rng.integers(1, 7, size=n)

is_one  = (rolls == 1)
is_odd  = (rolls % 2 == 1)

# Conditional via the definition: restrict to outcomes where the conditioning event holds
P_one_given_odd = np.mean(is_one & is_odd) / np.mean(is_odd)
print(f"P(1 | odd) ~ {P_one_given_odd:.4f}   (analytic 1/3 ≈ 0.3333)")

# Equivalently: count "1" among the rolls that were odd
odd_rolls = rolls[is_odd]
P_one_among_odd = np.mean(odd_rolls == 1)
print(f"empirical via subset: {P_one_among_odd:.4f}")


## Bayes' theorem

Probability's single most important identity — and the engine of every Bayesian model.

From the product rule above we have *two* expressions for `P(A \cap B)`:

$$
P(A | B) \, P(B) = P(B | A) \, P(A)
$$

Divide both sides by `P(B)` (when it's nonzero) and you get **Bayes' theorem**:

$$
\boxed{P(A | B) = \frac{P(B | A) \, P(A)}{P(B)}}
$$

In words: *the posterior is proportional to the likelihood times the prior, normalized.* Each piece has a name and a role:

| Symbol | Name | Reading |
|---|---|---|
| `P(A \| B)` | **posterior** | what we believe about A *after* seeing B |
| `P(B \| A)` | **likelihood** | how plausibly we'd see B *if* A were true |
| `P(A)` | **prior** | what we believed about A *before* seeing B |
| `P(B)` | **evidence** | total probability of seeing B (a normalizing constant) |

The evidence `P(B)` can be expanded using the **law of total probability**: for a partition `\{A_i\}` of the sample space,

$$
P(B) = \sum_i P(B | A_i) \, P(A_i)
$$

This is the "sum over all explanations weighted by their priors" expansion. We'll lean on it for the canonical Bayes' theorem example next.

**Classic example — disease testing.** A disease has prevalence 1% in the population. A test is 99% sensitive (catches the disease when present) and 99% specific (doesn't fire when absent). You test positive. What's the probability you actually have the disease?

Let `D` = "have disease," `+` = "test positive."

- Prior: `P(D) = 0.01`, `P(D^c) = 0.99`.
- Likelihoods: `P(+ | D) = 0.99`, `P(+ | D^c) = 0.01`.
- Evidence: `P(+) = P(+ | D) P(D) + P(+ | D^c) P(D^c) = 0.99 \cdot 0.01 + 0.01 \cdot 0.99 = 0.0198`.
- Posterior: `P(D | +) = P(+ | D) P(D) / P(+) = (0.99 \cdot 0.01) / 0.0198 \approx 0.5`.

Even with a "99% accurate" test, a positive result only means you have a **50%** chance of being sick — because the disease is rare. This is the **base rate fallacy**, and it's the most-cited Bayes example for a reason: human intuition badly underestimates the role of the prior.


In [ ]:
# Verify the disease test example by simulation
n = 1_000_000
have_disease  = rng.random(n) < 0.01                       # prior P(D) = 0.01
# Sensitivity 99% → if you have it, you test positive 99% of the time
# Specificity 99% → if you don't have it, you test negative 99% of the time
test_positive = np.where(
    have_disease,
    rng.random(n) < 0.99,     # P(+ | D)
    rng.random(n) < 0.01,     # P(+ | not D)
)

# Among positive testers, what fraction actually have the disease?
P_disease_given_pos = np.mean(have_disease[test_positive])
print(f"P(D | +) ~ {P_disease_given_pos:.4f}   (analytic ≈ 0.5)")

# Sanity checks of the inputs
print(f"\nempirical P(D)       = {np.mean(have_disease):.4f}")
print(f"empirical P(+ | D)   = {np.mean(test_positive[have_disease]):.4f}")
print(f"empirical P(+ | ¬D)  = {np.mean(test_positive[~have_disease]):.4f}")
print(f"empirical P(+)       = {np.mean(test_positive):.4f}")


## Independence

Two events `A` and `B` are **independent** if knowing one tells you nothing about the other:

$$
P(A | B) = P(A) \qquad \text{(equivalently, } P(B | A) = P(B)\text{)}
$$

Substituting back into the product rule gives the cleanest form of the definition:

$$
\boxed{P(A \cap B) = P(A) \, P(B) \quad \Longleftrightarrow \quad A, B \text{ independent}}
$$

**Two pitfalls people fall into.**

- **Independence is not the same as disjointness.** Disjoint events `A` and `B` *cannot* be independent (unless one has probability 0): if `B` happens, then `A` cannot, so knowing `B` changes everything about `A`. "Disjoint" is the *opposite* of "independent" in some sense.
- **Pairwise independence ≠ mutual independence.** Three events `A, B, C` can be pairwise independent without all three being jointly independent. The full definition of mutual independence requires `P(A \cap B \cap C) = P(A) P(B) P(C)` *and* all pairwise conditions.

**Conditional independence.** `A` and `B` are *conditionally independent given `C`* if `P(A \cap B | C) = P(A | C) P(B | C)`. This shows up everywhere in graphical models — d-separation, Bayes nets, naive Bayes. It's a much more useful property than plain independence, because pure independence is rare in real data, but *conditional* independence given the right covariates often holds.

**ML angle.** "Naive Bayes" is the assumption that all features are conditionally independent given the class. It's almost never true, but the assumption makes the math trivial and the model often works anyway — which is the punchline of half of practical ML.


In [ ]:
# Independence demo: rolling two fair dice
n = 200_000
die1 = rng.integers(1, 7, size=n)
die2 = rng.integers(1, 7, size=n)

A = (die1 == 6)              # "die 1 shows 6"
B = (die2 % 2 == 0)          # "die 2 is even"

P_A = np.mean(A)
P_B = np.mean(B)
P_A_and_B = np.mean(A & B)

print(f"P(A) = {P_A:.4f}   (analytic 1/6 ≈ 0.1667)")
print(f"P(B) = {P_B:.4f}   (analytic 1/2 = 0.5)")
print(f"P(A ∩ B)        = {P_A_and_B:.4f}")
print(f"P(A) · P(B)     = {P_A * P_B:.4f}")
print(f"\nIndependent? Two values agree → yes (different dice are independent).\n")

# Contrast: rolling ONE die and asking if 'is 6' and 'is even' are independent
A_same = (die1 == 6)
B_same = (die1 % 2 == 0)
print(f"Same die — P(is 6 ∩ is even) = {np.mean(A_same & B_same):.4f}")
print(f"          P(is 6) · P(is even) = {np.mean(A_same) * np.mean(B_same):.4f}")
print("Not independent — knowing the die is even raises P(is 6) from 1/6 to 1/3.")


## Where this appears in ML

Even before we talk about random variables, the axioms and Bayes' theorem are the language of probabilistic reasoning across ML.

- **Bayes' theorem in classifiers.** The *Bayes-optimal classifier* outputs `\arg\max_y P(y | x) = \arg\max_y P(x | y) P(y)`. Every classifier is approximating this — naive Bayes literally, neural classifiers implicitly.
- **Naive Bayes.** Apply Bayes' theorem with the conditional-independence-given-class assumption: `P(x_1, \dots, x_d | y) = \prod_i P(x_i | y)`. Cheap, fast, surprisingly competitive for text classification.
- **Posterior inference in Bayesian models.** Every Bayesian neural network, Gaussian process, latent Dirichlet allocation, etc., computes (or approximates) `P(\theta | D) \propto P(D | \theta) P(\theta)` — Bayes' rule one more time.
- **Bayesian model selection.** Compare models by their *evidence* `P(D | M)`, using Bayes' rule at the model level.
- **The base-rate fallacy in ML diagnostics.** Highly accurate medical or fraud-detection classifiers can still be useless on rare classes. Without addressing the base rate, "99% accuracy" can mean "always predict the majority class." Precision/recall trade-offs are a quantitative response to this.
- **Conditional independence in graphical models.** Bayesian networks, Markov random fields, HMMs — all built on conditional independence assumptions encoded in graph structure.
- **Causality.** The whole "causation vs correlation" discussion is about whether independencies are preserved under interventions. The probability axioms are the foundation; Judea Pearl's do-calculus is the upgrade.
- **A/B testing.** When you compute "the probability of seeing this result if A and B are actually equal," you are applying conditional probability under a null hypothesis.

Next notebook: **random variables and distributions** — we generalize "event probability" to "a whole function over outcomes," and meet the staple distributions you'll encounter in every probabilistic model.
